In [12]:
import os
import pandas as pd

# Define the input and output directories
input_dir = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Raw_Data_NSW_all'
output_dir = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Processed_Data_NSW_ALL'

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Loop through all CSV files in the input directory
for filename in os.listdir(input_dir):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_dir, filename)
        
        try:
            # Read the file, skipping the first two rows and specifying encoding
            df = pd.read_csv(file_path, skiprows=2, encoding='ISO-8859-1')

            # Strip any leading/trailing whitespace from column names
            df.columns = df.columns.str.strip()

            # Create a new list for column names: keep 'Date' and 'Time', extract the second word for others
            new_column_names = []
            for col in df.columns:
                if col in ['Date', 'Time']:
                    new_column_names.append(col)
                else:
                    # Extract the second word from the column name
                    second_word = col.split(' ')[1].strip()
                    new_column_names.append(second_word)
            
            # Assign the new column names to the DataFrame
            df.columns = new_column_names

            # Log the new column names
            print(f"New column names for {filename}: {df.columns.tolist()}")

            # Save the modified DataFrame to the output directory
            output_file = os.path.join(output_dir, filename)
            df.to_csv(output_file, index=False)
            print(f"Processed and saved {filename}")

        except KeyError as e:
            print(f"Key error in {filename}: {e}")
        except UnicodeDecodeError as e:
            print(f"Error reading {filename}: {e}")


New column names for RANDWICK_CSV_File_1728017408.csv: ['Date', 'Time', 'HUMID', 'NEPH', 'NO', 'NOX', 'OZONE', 'PM10', 'PM2.5', 'RAIN', 'SD1', 'SO2', 'TEMP', 'WDR', 'WGU', 'WSP']
Processed and saved RANDWICK_CSV_File_1728017408.csv
New column names for EARLWOOD_CSV_File_1728017283.csv: ['Date', 'Time', 'CO', 'CO_1', 'HUMID', 'NEPH', 'NO', 'NOX', 'OZONE', 'PM10', 'PM2.5', 'SD1', 'TEMP', 'WDR', 'WGU', 'WSP']
Processed and saved EARLWOOD_CSV_File_1728017283.csv
New column names for ALEXANDRIA_CSV_File_1728017147.csv: ['Date', 'Time', 'CO', 'HUMID', 'NEPH', 'NO', 'NOX', 'OZONE', 'PM10', 'PM2.5', 'SD1', 'SO2', 'TEMP', 'WDR', 'WSP']
Processed and saved ALEXANDRIA_CSV_File_1728017147.csv
New column names for CHULLORA_CSV_File_1728017242.csv: ['Date', 'Time', 'CO', 'CO_1', 'HUMID', 'NEPH', 'NO', 'NOX', 'OZONE', 'PM10', 'PM2.5', 'SD1', 'SO2', 'SOLAR', 'TEMP', 'WDR', 'WSP']
Processed and saved CHULLORA_CSV_File_1728017242.csv
New column names for LINDFIELD_CSV_File_1728017586.csv: ['Date', 'Time

In [11]:
import os
import pandas as pd
from pytz import timezone

# Define the input and output directories
input_dir = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Data/SydneyEast'
output_dir = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Processed_Data'

# Set the timezone (for example, Sydney timezone with UTC+11)
sydney_tz = timezone('Australia/Sydney')

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Function to adjust '24:00' to '00:00' and increment the date
def fix_time_and_date(date_series, time_series):
    # Replace '24:00' with '00:00'
    time_series_fixed = time_series.replace('24:00', '00:00')
    
    # Convert Date and Time into datetime
    datetime_series = pd.to_datetime(date_series + ' ' + time_series_fixed, format='%d/%m/%Y %H:%M', errors='coerce')
    
    # Increment the date for times that were originally '24:00'
    datetime_series += pd.to_timedelta(time_series.str.contains('24:00') * 1, unit='d')
    
    return datetime_series

# Loop through all CSV files in the input directory
for filename in os.listdir(input_dir):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_dir, filename)
        
        try:
            # Read the file, skipping the first two rows and specifying encoding
            df = pd.read_csv(file_path, skiprows=2, encoding='ISO-8859-1')

            # Strip any leading/trailing whitespace from column names
            df.columns = df.columns.str.strip()

            # Fix Date and Time columns and create the datetime column
            df['datetime'] = fix_time_and_date(df['Date'], df['Time'])

            # Handle ambiguous times during DST transitions by setting ambiguous=False (standard time)
            df['datetime'] = df['datetime'].dt.tz_localize(
                sydney_tz, 
                ambiguous=False,  # Resolves ambiguity by defaulting to standard time
                nonexistent='shift_forward'  # Handles nonexistent times during DST transitions by shifting forward
            )

            # Drop the original Date and Time columns
            df = df.drop(columns=['Date', 'Time'])

            # Assign new column names based on the second word of other columns
            new_column_names = ['datetime']
            for col in df.columns[1:]:  # Skipping the 'datetime' column
                second_word = col.split(' ')[1].strip() if ' ' in col else col
                new_column_names.append(second_word)

            # Assign the new column names to the DataFrame
            df.columns = new_column_names

            # Log the new column names
            print(f"New column names for {filename}: {df.columns.tolist()}")

            # Save the modified DataFrame to the output directory
            output_file = os.path.join(output_dir, filename)
            df.to_csv(output_file, index=False)
            print(f"Processed and saved {filename}")

        except KeyError as e:
            print(f"Key error in {filename}: {e}")
        except UnicodeDecodeError as e:
            print(f"Error reading {filename}: {e}")


New column names for RANDWICK_CSV_File_1728017408.csv: ['datetime', 'NEPH', 'NO', 'NOX', 'OZONE', 'PM10', 'PM2.5', 'RAIN', 'SD1', 'SO2', 'TEMP', 'WDR', 'WGU', 'WSP', 'datetime']
Processed and saved RANDWICK_CSV_File_1728017408.csv
New column names for EARLWOOD_CSV_File_1728017283.csv: ['datetime', 'CO_1', 'HUMID', 'NEPH', 'NO', 'NOX', 'OZONE', 'PM10', 'PM2.5', 'SD1', 'TEMP', 'WDR', 'WGU', 'WSP', 'datetime']
Processed and saved EARLWOOD_CSV_File_1728017283.csv
New column names for ALEXANDRIA_CSV_File_1728017147.csv: ['datetime', 'HUMID', 'NEPH', 'NO', 'NOX', 'OZONE', 'PM10', 'PM2.5', 'SD1', 'SO2', 'TEMP', 'WDR', 'WSP', 'datetime']
Processed and saved ALEXANDRIA_CSV_File_1728017147.csv
New column names for CHULLORA_CSV_File_1728017242.csv: ['datetime', 'CO_1', 'HUMID', 'NEPH', 'NO', 'NOX', 'OZONE', 'PM10', 'PM2.5', 'SD1', 'SO2', 'SOLAR', 'TEMP', 'WDR', 'WSP', 'datetime']
Processed and saved CHULLORA_CSV_File_1728017242.csv
New column names for LINDFIELD_CSV_File_1728017586.csv: ['datetim

In [3]:
import os
import pandas as pd

# Define the input and output directories
input_dir = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Raw_Data_NSW_all'
output_dir = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Processed_Data_NSW_ALL'

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Read the station info to get the mapping between "Number" and "SiteName"
station_info_path = '/mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Station_info.csv'
station_info = pd.read_csv(station_info_path, encoding='ISO-8859-1')

# Create a dictionary to map "Number" to "SiteName"
station_mapping = dict(zip(station_info['Number'].astype(str), station_info['SiteName']))

# Loop through all CSV files in the input directory
for filename in os.listdir(input_dir):
    if filename.endswith('.csv'):
        file_path = os.path.join(input_dir, filename)
        
        try:
            # Read the file, skipping the first two rows and specifying encoding
            df = pd.read_csv(file_path, skiprows=2, encoding='ISO-8859-1')

            # Strip any leading/trailing whitespace from column names
            df.columns = df.columns.str.strip()

            # Create a new list for column names: keep 'Date' and 'Time', extract the second word for others
            new_column_names = []
            site_name = None  # Initialize site name

            for col in df.columns:
                if col in ['Date', 'Time']:
                    new_column_names.append(col)
                else:
                    # Extract the first word (number) and map it to the site name
                    number = col.split(' ')[0].strip()
                    second_word = col.split(' ')[1].strip()

                    # Use the number to get the corresponding site name from the station info
                    site_name = station_mapping.get(number, None)

                    if site_name is not None:
                        new_column_names.append(second_word)
                    else:
                        print(f"SiteName not found for Number: {number} in {filename}")
            
            # Assign the new column names to the DataFrame
            df.columns = new_column_names

            # Log the new column names
            print(f"New column names for {filename}: {df.columns.tolist()}")

            # Save the modified DataFrame to the output directory with the site name
            if site_name:
                output_file = os.path.join(output_dir, f"{site_name}_AQMS_Processed.csv")
                df.to_csv(output_file, index=False)
                print(f"Processed and saved {filename} as {output_file}")
            else:
                print(f"Site name could not be determined for {filename}, skipping save.")

        except KeyError as e:
            print(f"Key error in {filename}: {e}")
        except UnicodeDecodeError as e:
            print(f"Error reading {filename}: {e}")


New column names for CSV_File_1728127295.csv: ['Date', 'Time', 'H2O', 'HUMID', 'NO', 'NOX', 'PM10', 'PM2.5', 'RAIN', 'TEMP', 'WDR', 'WGU', 'WSP']
Processed and saved CSV_File_1728127295.csv as /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Processed_Data_NSW_ALL/GUNNEDAH_AQMS_Processed.csv
New column names for CSV_File_1728129035.csv: ['Date', 'Time', 'H2O', 'HUMID', 'NO', 'NOX', 'PM10', 'PM2.5', 'RAIN', 'SO2', 'TEMP', 'WDR', 'WGU', 'WSP']
Processed and saved CSV_File_1728129035.csv as /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Processed_Data_NSW_ALL/BERESFIELD_AQMS_Processed.csv
New column names for CSV_File_1728126500.csv: ['Date', 'Time', 'CO', 'H2O', 'HUMID', 'NO', 'NOX', 'PM10', 'PM2.5', 'RAIN', 'SO2', 'TEMP', 'WDR', 'WGU', 'WSP']
Processed and saved CSV_File_1728126500.csv as /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Processed_Data_NSW_ALL/WYONG_AQMS_Processed.csv
New column names for CSV_File_1728126278.csv: ['Date', 'Time', 'CO', 'CO2',

/home/ahmedmas/.local/lib/python3.6/site-packages/IPython/core/interactiveshell.py:3072: DtypeWarning: Columns (2,3,4,5,6,7,8,9,10) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


Processed and saved CSV_File_1728127580.csv as /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Processed_Data_NSW_ALL/BRADFIELD-HIGHWAY_AQMS_Processed.csv
New column names for CSV_File_1728126968.csv: ['Date', 'Time', 'H2O', 'HUMID', 'NO', 'NOX', 'PM10', 'PM2.5', 'RAIN', 'SO2', 'TEMP', 'WDR', 'WGU', 'WSP']
Processed and saved CSV_File_1728126968.csv as /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Processed_Data_NSW_ALL/BERESFIELD_AQMS_Processed.csv
New column names for CSV_File_1728125585.csv: ['Date', 'Time', 'H2O', 'HUMID', 'NO', 'NOX', 'PM10', 'PM2.5', 'RAIN', 'TEMP', 'WDR', 'WGU', 'WSP']
Processed and saved CSV_File_1728125585.csv as /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Input_Data/Processed_Data_NSW_ALL/GOULBURN_AQMS_Processed.csv
New column names for CSV_File_1728127878.csv: ['Date', 'Time', 'H2O', 'HUMID', 'PM10', 'RAIN', 'TEMP', 'WDR', 'WGU', 'WSP']
Processed and saved CSV_File_1728127878.csv as /mnt/scratch_lustre/barthelx/Masrur/Nawcasting/Inpu